# 03 - Data Cleaning and Preprocessing

This notebook converts the interim merged dataset into a cleaned analytical dataset while preserving auditability.

## Purpose

The objective is not to make the data look perfect. The objective is to make the dataset **usable, explainable, and defensible** for Canadian retail credit-risk analytics.

This notebook:

- fixes source-level formatting issues;
- converts obvious placeholder values into explicit `Unknown` categories;
- creates missingness and quality flags instead of silently deleting rows;
- preserves the modelling grain established in Notebook 01;
- documents which variables require leakage, fairness, or governance review before modelling.

Professional principle: in credit risk, missingness and data-quality defects can carry risk signal. We flag them first and decide later whether they should be used in the model.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from credit_risk.config import INTERIM_DIR, PROCESSED_DIR, TABLE_DIR
from credit_risk.data.cleaning import clean_credit_risk_dataset

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INTERIM_PATH = INTERIM_DIR / "credit_risk_merged_interim.csv"
PROCESSED_PATH = PROCESSED_DIR / "credit_risk_cleaned.csv"
INTERIM_PATH

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/interim/credit_risk_merged_interim.csv')

## Load interim analytical dataset

In [2]:
df = pd.read_csv(INTERIM_PATH, low_memory=False)

pre_clean_overview = {
    "row_count": df.shape[0],
    "column_count": df.shape[1],
    "full_duplicate_row_count": int(df.duplicated().sum()),
    "record_key_duplicate_count": int(df.duplicated(["user_id", "record_sequence"]).sum()),
    "target_default_rate": float(df["defaulter"].mean()),
}
pre_clean_overview

{'row_count': 134417,
 'column_count': 25,
 'full_duplicate_row_count': 0,
 'record_key_duplicate_count': 0,
 'target_default_rate': 0.0904126710163149}

## Cleaning policy

These rules are designed for a portfolio project that mirrors financial-institution practice: minimal row deletion, traceable transformations, and clear governance flags.

In [3]:
cleaning_policy = pd.DataFrame(
    [
        {
            "issue": "Duplicate user_id values",
            "decision": "Preserve user_id + record_sequence grain",
            "business_reason": "Repeated borrowers can represent multiple records; direct user_id merge would inflate the portfolio.",
        },
        {
            "issue": "Missing loan amount and non-positive loan amount",
            "decision": "Create amount quality flags and set non-positive amount to missing",
            "business_reason": "Amount is core exposure information. Do not drop records before assessing missingness impact.",
        },
        {
            "issue": "Employment fields with high missingness",
            "decision": "Create missing flags and fill category as Unknown",
            "business_reason": "Employment missingness can be informative and should remain auditable.",
        },
        {
            "issue": "industry and work_experience placeholder value 0",
            "decision": "Treat 0 as Unknown and retain placeholder flags",
            "business_reason": "These are not credible business categories; they indicate unavailable source information.",
        },
        {
            "issue": "Repayment fields available in dataset",
            "decision": "Keep for portfolio monitoring, exclude from baseline default prediction",
            "business_reason": "Repayment amounts may be observed after origination and can cause target leakage.",
        },
        {
            "issue": "Sensitive/proxy variables",
            "decision": "Keep for audit and monitoring, not for baseline model training",
            "business_reason": "Fields such as gender, marital status, geography, and social profile require governance review.",
        },
    ]
)
cleaning_policy

,issue,decision,business_reason
0,Duplicate user_id values,Preserve user_id + record_sequence grain,Repeated borrowers can represent multiple reco...
1,Missing loan amount and non-positive loan amount,Create amount quality flags and set non-positi...,Amount is core exposure information. Do not dr...
2,Employment fields with high missingness,Create missing flags and fill category as Unknown,Employment missingness can be informative and ...
3,industry and work_experience placeholder value 0,Treat 0 as Unknown and retain placeholder flags,These are not credible business categories; th...
4,Repayment fields available in dataset,"Keep for portfolio monitoring, exclude from ba...",Repayment amounts may be observed after origin...
5,Sensitive/proxy variables,"Keep for audit and monitoring, not for baselin...","Fields such as gender, marital status, geograp..."


## Apply cleaning pipeline

In [4]:
cleaning_result = clean_credit_risk_dataset(df)
cleaned_df = cleaning_result.cleaned

audit_summary = cleaning_result.audit_summary
flag_summary = cleaning_result.flag_summary
model_feature_policy = cleaning_result.model_feature_policy

audit_summary.to_csv(TABLE_DIR / "cleaning_audit_summary.csv", index=False)
flag_summary.to_csv(TABLE_DIR / "cleaning_flag_summary.csv", index=False)
model_feature_policy.to_csv(TABLE_DIR / "model_feature_policy.csv", index=False)

audit_summary

,metric,value
0,rows_before,"134,417.0000"
1,rows_after,"134,417.0000"
2,columns_before,25.0000
3,columns_after,46.0000
4,target_default_rate_after,0.0904
5,full_duplicate_rows_after,0.0000
6,record_key_duplicate_count_after,0.0000


## Data-quality flag summary

In [5]:
flag_summary

,flag,flagged_row_count,flagged_row_pct
14,has_broad_data_quality_issue,116527,86.6907
7,industry_missing_flag,113583,84.5005
8,work_experience_missing_flag,113583,84.5005
2,industry_placeholder_zero_flag,113579,84.4975
3,work_experience_placeholder_zero_flag,113579,84.4975
5,employment_type_missing_flag,79686,59.2827
6,tier_of_employment_missing_flag,79686,59.2827
9,married_missing_flag,45084,33.5404
10,social_profile_missing_flag,44846,33.3633
11,is_verified_missing_flag,33507,24.9277


### Interpretation

The high `industry` and `work_experience` placeholder rates mean those fields should not be treated as reliable granular predictors without grouping or exclusion. This is a useful governance finding, not a project failure.

For modelling, the **core** data-quality issue rate is more important than the broad placeholder rate because it focuses on exposure amount and repayment consistency issues.

## Missingness after cleaning

In [6]:
post_clean_missingness = pd.DataFrame(
    {
        "column": cleaned_df.columns,
        "dtype": [str(dtype) for dtype in cleaned_df.dtypes],
        "missing_count": cleaned_df.isna().sum().values,
        "missing_pct": (cleaned_df.isna().mean().values * 100).round(4),
        "unique_values": [cleaned_df[col].nunique(dropna=True) for col in cleaned_df.columns],
    }
).sort_values("missing_pct", ascending=False)

post_clean_missingness.to_csv(TABLE_DIR / "post_clean_missingness.csv", index=False)
post_clean_missingness.head(20)

,column,dtype,missing_count,missing_pct,unique_values
2,amount,float64,27638,20.5614,86156
40,interest_to_amount_ratio,float64,27638,20.5614,106695
39,principal_to_amount_ratio,float64,27638,20.5614,106248
38,payment_to_amount_ratio,float64,27638,20.5614,106764
37,loan_to_income_ratio,float64,27638,20.5614,104531
0,user_id,int64,0,0.0000,133752
33,work_experience_missing_flag,int32,0,0.0000,2
26,amount_non_positive_flag,int32,0,0.0000,2
27,industry_placeholder_zero_flag,int32,0,0.0000,2
28,work_experience_placeholder_zero_flag,int32,0,0.0000,2


## Cleaned categorical review

In [7]:
categorical_cols = cleaned_df.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()

categorical_review = pd.DataFrame(
    {
        "column": categorical_cols,
        "missing_count": [cleaned_df[col].isna().sum() for col in categorical_cols],
        "missing_pct": [(cleaned_df[col].isna().mean() * 100).round(4) for col in categorical_cols],
        "unique_values": [cleaned_df[col].nunique(dropna=True) for col in categorical_cols],
        "top_value": [cleaned_df[col].value_counts(dropna=False).index[0] for col in categorical_cols],
        "top_value_share_pct": [round(cleaned_df[col].value_counts(dropna=False, normalize=True).iloc[0] * 100, 4) for col in categorical_cols],
    }
).sort_values(["unique_values", "top_value_share_pct"], ascending=[False, False])

categorical_review.to_csv(TABLE_DIR / "post_clean_categorical_review.csv", index=False)
categorical_review

,column,missing_count,missing_pct,unique_values,top_value,top_value_share_pct
3,industry,0,0.0000,12974,Unknown,84.5005
9,pincode,0,0.0000,844,XX112X,1.1784
4,role,0,0.0000,46,KHMbckjadbckIFGCASEWdkcndwkcnCCM,15.5285
2,tier_of_employment,0,0.0000,8,Unknown,59.2827
5,work_experience,0,0.0000,7,Unknown,84.5005
0,loan_category,0,0.0000,7,Consolidation,60.1620
8,home,0,0.0000,4,Mortgage,48.7215
11,is_verified,0,0.0000,4,Not Verified,25.0876
1,employment_type,0,0.0000,3,Unknown,59.2827
7,married,0,0.0000,3,Unknown,33.5404


## Key cleaned category distributions

In [8]:
for col in [
    "loan_category",
    "employment_type",
    "tier_of_employment",
    "work_experience",
    "home",
    "is_verified",
]:
    print(f"\n{col}")
    display(cleaned_df[col].value_counts(dropna=False).head(15).to_frame("row_count"))


loan_category


,row_count
loan_category,
Consolidation,80868
Credit Card,31668
Home,8503
Other,8225
Business,2236
Car,1538
Medical,1379



employment_type


,row_count
employment_type,
Unknown,79686
Salaried,44817
Self-Employed,9914



tier_of_employment


...

,row_count
work_experience,
Unknown,113583
5-10,8132
10+,6486
1-2,1843
2-3,1538
<1,1443
3-5,1392



home


,row_count
home,
Mortgage,65490
Rent,56442
Own,12397
Other/None,88



is_verified


,row_count
is_verified,
Not Verified,33722
Source Verified,33630
Verified,33558
Unknown,33507


## Numeric reasonableness review after cleaning

In [9]:
numeric_review_cols = [
    "amount",
    "interest_rate",
    "tenure_years",
    "total_income_pa",
    "delinq_2yrs",
    "number_of_loans",
    "loan_to_income_ratio",
    "payment_to_amount_ratio",
    "principal_to_amount_ratio",
]

numeric_review = cleaned_df[numeric_review_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numeric_review.to_csv(TABLE_DIR / "post_clean_numeric_review.csv")
numeric_review

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
amount,"106,779.0000","137,653.6574","157,599.9759",5.0000,"3,570.3400","8,816.0000","28,520.5000","76,549.0000","205,482.5000","422,275.3000","659,305.3800","8,000,078.0000"
interest_rate,"134,417.0000",12.0248,3.8793,5.4200,5.4500,5.9800,9.1700,11.8400,14.2800,18.9600,22.1300,23.5400
tenure_years,"134,417.0000",4.5231,0.8789,4.0000,4.0000,4.0000,4.0000,4.0000,6.0000,6.0000,6.0000,6.0000
total_income_pa,"134,417.0000","72,597.9750","56,100.0513","4,000.0000","18,312.3200","27,000.0000","45,000.0000","62,000.0000","87,000.0000","150,000.0000","250,000.0000","7,141,778.0000"
delinq_2yrs,"134,417.0000",0.2831,0.7992,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,4.0000,22.0000
number_of_loans,"134,417.0000",0.0056,0.0990,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5.0000
loan_to_income_ratio,"106,779.0000",1.8033,1.5623,0.0001,0.0728,0.1866,0.5233,1.2359,2.8413,4.7644,6.3368,19.6375
payment_to_amount_ratio,"106,779.0000",0.3692,20.0826,0.0000,0.0061,0.0133,0.0381,0.0971,0.2568,0.8911,1.8760,"6,251.6800"
principal_to_amount_ratio,"106,779.0000",0.2750,10.2366,0.0000,0.0037,0.0085,0.0258,0.0693,0.1883,0.7209,1.5611,"2,819.1720"


## Target and data-quality relationship

In [10]:
quality_by_target = (
    cleaned_df
    .groupby("defaulter")
    .agg(
        row_count=("defaulter", "size"),
        default_rate=("defaulter", "mean"),
        amount_missing_rate=("amount_missing_flag", "mean"),
        core_quality_issue_rate=("has_core_data_quality_issue", "mean"),
        broad_quality_issue_rate=("has_broad_data_quality_issue", "mean"),
        median_amount=("amount", "median"),
        median_income=("total_income_pa", "median"),
    )
    .reset_index()
)

for col in ["amount_missing_rate", "core_quality_issue_rate", "broad_quality_issue_rate", "default_rate"]:
    quality_by_target[col] = quality_by_target[col] * 100

quality_by_target.to_csv(TABLE_DIR / "quality_by_target.csv", index=False)
quality_by_target

,defaulter,row_count,default_rate,amount_missing_rate,core_quality_issue_rate,broad_quality_issue_rate,median_amount,median_income
0,0,122264,0.0000,18.9598,21.2074,87.5409,"78,520.0000","62,500.0000"
1,1,12153,100.0000,36.6741,37.0773,78.1371,"58,169.5000","55,000.0000"


## Model feature policy

This table is carried into Notebook 05 and Notebook 06. It prevents accidental leakage and shows hiring managers that the project is governed, not just optimized for a high score.

In [11]:
model_feature_policy

,column,recommended_use,reason
0,user_id,exclude_from_model,Identifier; useful for joins and audit only.
1,record_sequence,exclude_from_model,Technical merge key; no business meaning.
2,defaulter,target_only,Target variable; never used as predictor.
3,total_payment,exclude_from_baseline_model,May include post-origination repayment behavio...
4,received_principal,exclude_from_baseline_model,May be observed after the lending decision or ...
5,interest_received,exclude_from_baseline_model,May be post-outcome repayment information.
6,gender,fairness_audit_only,Sensitive/proxy field; use for bias diagnostic...
7,married,fairness_audit_or_exclude,Household status proxy; include only with clea...
8,pincode,portfolio_monitoring_or_exclude,Masked geographic field; can encode socioecono...
9,social_profile,exclude_or_governance_review,Unclear business meaning and potential behavio...


## Save cleaned processed dataset

In [12]:
cleaned_df.to_csv(PROCESSED_PATH, index=False)
PROCESSED_PATH

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/processed/credit_risk_cleaned.csv')

## Data cleaning decisions to carry forward

1. The row count remains unchanged at the safe observation grain.
2. `amount` remains missing where the source is missing or invalid; later model pipelines should impute inside cross-validation, not before splitting.
3. Employment missingness is retained through explicit flags and `Unknown` categories.
4. `industry` and `work_experience` are heavily affected by placeholder zeros; they require careful treatment before modelling.
5. `total_payment`, `received_principal`, and `interest_received` are useful for portfolio monitoring but should be excluded from the baseline early-warning model because of timing/leakage risk.
6. Sensitive and proxy variables are preserved for audit but should not be used in the first champion model.

Next notebook: **04 - Credit Risk EDA and Portfolio Monitoring**.